<a href="https://colab.research.google.com/github/Daniela-Alves2004/data-mining/blob/main/Aula17_06_sem_Data_Leakage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

In [3]:
# Consolidando importações e preparação inicial dos dados para garantir que todas as variáveis e funções estejam definidas.

# Importações necessárias
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# Carregar a base de dados
file_path = "healthcare-dataset-stroke-data.csv"
try:
    df = pd.read_csv(file_path)
    print(f"Arquivo '{file_path}' carregado com sucesso.")
except FileNotFoundError:
    print(f"Erro: O arquivo '{file_path}' não foi encontrado.")
    print("Por favor, faça o upload do arquivo para o ambiente do Colab ou verifique o caminho.")
    raise FileNotFoundError(f"Certifique-se de que '{file_path}' esteja no diretório correto antes de prosseguir.")

# Ajustes gerais
# i) Remover o atributo id
df = df.drop(columns=["id"])

# ii) Separar atributos e classes
X = df.drop(columns=['stroke'])
y = df["stroke"]

# iii) Guardar as colunas numéricas e categóricas
numeric_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Configuração inicial concluída. 'df', 'X', 'y', 'numeric_cols' e 'categorical_cols' estão definidos.")

Arquivo 'healthcare-dataset-stroke-data.csv' carregado com sucesso.
Configuração inicial concluída. 'df', 'X', 'y', 'numeric_cols' e 'categorical_cols' estão definidos.


In [4]:
# Carregar a base de dados
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

In [5]:
# 1. Separar atributos (X) e classes (y) no início (já feito em ePMz1qAszP9O)
# X = df.drop(columns=['stroke'])
# y = df["stroke"]

# 2. Separar dados em treino e teste ANTES do pré-processamento
X_train_leak_fix, X_test_leak_fix, y_train_leak_fix, y_test_leak_fix = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=2,
    stratify=y
)

print(f"X_train_leak_fix shape: {X_train_leak_fix.shape}")
print(f"X_test_leak_fix shape: {X_test_leak_fix.shape}")
print(f"y_train_leak_fix shape: {y_train_leak_fix.shape}")
print(f"y_test_leak_fix shape: {y_test_leak_fix.shape}")

X_train_leak_fix shape: (3577, 10)
X_test_leak_fix shape: (1533, 10)
y_train_leak_fix shape: (3577,)
y_test_leak_fix shape: (1533,)


In [6]:
# 3. Identificar colunas numéricas e categóricas (já feito em B2bLe7MFzfp2)
# numeric_cols = X.select_dtypes(include=["int64","float64"]).columns.tolist()
# categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

# Criar cópias para o pré-processamento
X_train_processed = X_train_leak_fix.copy()
X_test_processed = X_test_leak_fix.copy()

In [7]:
# 4. Tratar valores faltantes (SimpleImputer) - APLICAR EM TRAIN E TEST
imputer_num_leak_fix = SimpleImputer(strategy="mean")

# Fit e transform no treino
X_train_processed[numeric_cols] = imputer_num_leak_fix.fit_transform(X_train_processed[numeric_cols])
# Apenas transform no teste
X_test_processed[numeric_cols] = imputer_num_leak_fix.transform(X_test_processed[numeric_cols])

print("Valores nulos em X_train_processed (numéricas) após imputação:")
display(X_train_processed[numeric_cols].isna().sum())
print("Valores nulos em X_test_processed (numéricas) após imputação:")
display(X_test_processed[numeric_cols].isna().sum())

Valores nulos em X_train_processed (numéricas) após imputação:


,0
age,0
hypertension,0
heart_disease,0
avg_glucose_level,0
bmi,0


Valores nulos em X_test_processed (numéricas) após imputação:


,0
age,0
hypertension,0
heart_disease,0
avg_glucose_level,0
bmi,0


In [8]:
# 5. Normalizar os dados (StandardScaler) - APLICAR EM TRAIN E TEST
scaler_leak_fix = StandardScaler()

# Fit e transform no treino
X_train_processed[numeric_cols] = scaler_leak_fix.fit_transform(X_train_processed[numeric_cols])
# Apenas transform no teste
X_test_processed[numeric_cols] = scaler_leak_fix.transform(X_test_processed[numeric_cols])

print("X_train_processed (numéricas) após escalonamento:")
display(X_train_processed[numeric_cols].head())
print("X_test_processed (numéricas) após escalonamento:")
display(X_test_processed[numeric_cols].head())

X_train_processed (numéricas) após escalonamento:


,age,hypertension,heart_disease,avg_glucose_level,bmi
4827,0.603298,-0.325669,-0.236189,0.004606,4.381145e-01
3028,1.047766,3.070598,-0.236189,-0.454760,-4.638536e-16
1093,-1.915950,-0.325669,-0.236189,-1.110176,-1.690065e+00
490,0.425510,-0.325669,-0.236189,-0.902620,-5.800287e-03
1055,0.603298,-0.325669,-0.236189,0.124094,3.336640e-01


X_test_processed (numéricas) após escalonamento:


,age,hypertension,heart_disease,avg_glucose_level,bmi
4557,-1.174576,-0.325669,-0.236189,-0.536853,2.057098
3212,0.247723,3.070598,-0.236189,-0.948645,-0.736954
2427,-0.285639,-0.325669,-0.236189,-0.634214,-0.985024
2326,-0.285639,-0.325669,-0.236189,-0.374438,-0.632504
739,0.514404,-0.325669,-0.236189,-1.121682,-0.240814


In [9]:
# 6. Transformar as colunas categóricas com OHE - APLICAR EM TRAIN E TEST
encoder_leak_fix = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # Adiciona handle_unknown para robustez

# Fit e transform no treino
X_cat_train_encoded = encoder_leak_fix.fit_transform(X_train_processed[categorical_cols])
# Apenas transform no teste
X_cat_test_encoded = encoder_leak_fix.transform(X_test_processed[categorical_cols])

# Seleciona o nome das colunas geradas
cat_names_leak_fix = encoder_leak_fix.get_feature_names_out(categorical_cols)

# Transformar em DataFrames
X_cat_train_encoded_df = pd.DataFrame(
    X_cat_train_encoded,
    columns=cat_names_leak_fix,
    index=X_train_processed.index
)
X_cat_test_encoded_df = pd.DataFrame(
    X_cat_test_encoded,
    columns=cat_names_leak_fix,
    index=X_test_processed.index
)

print("Colunas categóricas codificadas no treino:")
display(X_cat_train_encoded_df.head())
print("Colunas categóricas codificadas no teste:")
display(X_cat_test_encoded_df.head())

Colunas categóricas codificadas no treino:


,gender_Female,gender_Male,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
4827,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3028,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1093,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
490,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1055,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


Colunas categóricas codificadas no teste:


,gender_Female,gender_Male,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
4557,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3212,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2427,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2326,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
739,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [10]:
# 7. Ajustar o dataframe de atributos, removendo as features categóricas originais e concatenar com as novas features categóricas

X_train_final = X_train_processed.drop(columns=categorical_cols)
X_train_final = pd.concat([X_train_final, X_cat_train_encoded_df], axis=1)

X_test_final = X_test_processed.drop(columns=categorical_cols)
X_test_final = pd.concat([X_test_final, X_cat_test_encoded_df], axis=1)

print(f"X_train_final shape: {X_train_final.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

print("X_train_final head:")
display(X_train_final.head())
print("X_test_final head:")
display(X_test_final.head())

X_train_final shape: (3577, 20)
X_test_final shape: (1533, 20)
X_train_final head:


,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
4827,0.603298,-0.325669,-0.236189,0.004606,4.381145e-01,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3028,1.047766,3.070598,-0.236189,-0.454760,-4.638536e-16,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1093,-1.915950,-0.325669,-0.236189,-1.110176,-1.690065e+00,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
490,0.425510,-0.325669,-0.236189,-0.902620,-5.800287e-03,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1055,0.603298,-0.325669,-0.236189,0.124094,3.336640e-01,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


X_test_final head:


,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
4557,-1.174576,-0.325669,-0.236189,-0.536853,2.057098,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3212,0.247723,3.070598,-0.236189,-0.948645,-0.736954,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2427,-0.285639,-0.325669,-0.236189,-0.634214,-0.985024,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2326,-0.285639,-0.325669,-0.236189,-0.374438,-0.632504,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
739,0.514404,-0.325669,-0.236189,-1.121682,-0.240814,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [11]:
# 8. Treinar o modelo (DecisionTree) com os dados corrigidos
modelo_leak_fix = DecisionTreeClassifier(random_state=2)
modelo_leak_fix.fit(X_train_final, y_train_leak_fix)

DecisionTreeClassifier(random_state=2)

In [12]:
# 9. Avaliar o modelo corrigido
y_pred_leak_fix = modelo_leak_fix.predict(X_test_final)

print("Relatório de Classificação (Correção Data Leakage):")
print(classification_report(y_test_leak_fix, y_pred_leak_fix))

Relatório de Classificação (Correção Data Leakage):
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1458
           1       0.18      0.24      0.21        75

    accuracy                           0.91      1533
   macro avg       0.57      0.59      0.58      1533
weighted avg       0.92      0.91      0.92      1533

